# 2 · Questionnaire detail  `[EVAL]`

**Level 2 of the drill-down: inside each questionnaire.** One uniform section per rubric, each in the same small-multiples style as `trajectories_all_metrics`: a per-item/per-component **detail grid** (trajectories across iterations, arms overlaid) + **"which items drive the change"** delta bars at each arm's **final AND best** iteration + a merged table (`target` column). Likert-item rubrics (Q1, Q2, WAI-SR, CSQ-8, MI-SAT) use the generic item loader; MITI / PCT / MICI decompose into their oracle-annotated behaviour rates + globals. Exports → `results/<VIEW>/figures|tables/2_questionnaires/` (headline copies of the MITI/MICI grids → `0_headline/`).

Global scores → `1_Outcomes`; the cross-cutting validity/hacking synthesis these details feed → `3_Validity_and_Hacking`; stats → `7_Stats`.

In [ ]:
import sys, os; sys.path.insert(0, os.path.abspath("."))
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
pd.set_option("display.width", 185, "display.max_columns", 50)
import eda_analysis
from eda_analysis import stats, behavior, figures, plots

# ╔═══ VIEW — the one knob ════════════════════════════════════════════════════════╗
# "all" = every arm | "L0" = K=0 arms (PTO_LA0/GRPO_LA0) | "L5" = K=5 arms (thin, paused).
# Sets BOTH the arm filter AND the results root -> results/<VIEW>/figures|tables/<group>/.
# Edit the default for interactive use; render_views.py overrides it via the EDA_VIEW env var.
VIEW = os.environ.get("EDA_VIEW", "L0")

cfg = eda_analysis.EdaConfig(
    view=VIEW,                             # arm filter + results/<VIEW>/ root
    export_group="2_questionnaires",       # topic family = this notebook's number
    selection="all",
    focus_arms=None, focus_metric="Q1Q2",
)
S = eda_analysis.notebook_setup(cfg)
# Best iteration per arm (own training oracle — the checkpoint you'd select); drives the *_best deltas.
BEST = eda_analysis.best_iteration_by_arm(S.SCORES)
print("best iteration per arm:", BEST)

In [ ]:
# The uniform Likert-item pattern (Q1 / Q2 / WAI-SR / CSQ-8 / MI-SAT): detail grid +
# final/best delta bars + one merged table. Q2 keeps its face-content group colors (q2_style).
def item_section(qname, slug, q2_style=False):
    IT = eda_analysis.load_items(qname, S.ARMS)
    if IT.empty:
        print(f"{qname} not scored yet — run Run_Eval.ipynb.")
        return IT
    fig = plots.item_trajectory_grid(
        IT, palette=S.PALETTE,
        title=f"{eda_analysis.display_label(qname)} — per-item trajectories")
    if fig:
        eda_analysis.save_fig(fig, f"{slug}_detail_grid",
                              caption=f"{qname}: per-item mean +/- 95% CI across iterations, arms overlaid — "
                                      f"the item-level drill-down behind the {qname} global score.")
        plt.show()
    deltas = {}
    for tgt, note, tmap in (("final", "final iteration", None),
                            ("best", "best iteration (own oracle)", BEST)):
        D = (stats.q2_item_endpoint_deltas(IT, target_iter_by_arm=tmap) if q2_style
             else stats.item_endpoint_deltas(IT, target_iter_by_arm=tmap))
        deltas[tgt] = D
        figd = (plots.q2_item_delta_bars(D) if q2_style
                else plots.item_delta_bars(D, title=f"{qname} — which items drive the change ({note})",
                                           xlabel="Δ item mean vs base (1–5 scale)"))
        if figd:
            eda_analysis.save_fig(figd, f"{slug}_item_deltas_{tgt}",
                                  caption=f"Δ vs base per {qname} item at each arm's {note} "
                                          "(one panel per arm, shared Δ scale).")
            plt.show()
    T = pd.concat([deltas["final"].assign(target="final"), deltas["best"].assign(target="best")],
                  ignore_index=True)
    T = T[["target"] + [c for c in T.columns if c != "target"]]
    eda_analysis.save_table(T, f"{slug}_item_deltas",
                            caption=f"Per (arm, {qname} item): base/target means + Δ at the final AND best "
                                    "iteration (target column).")
    return IT


# The behaviour-frame pattern (MITI / PCT / MICI): melt the by-iter frame into (item, score)
# rows so the same stats.item_endpoint_deltas drives the final/best delta bars + merged table.
def detail_deltas(det, slug, qname, xlabel):
    melted = det.melt(id_vars=["arm", "method", "K", "iteration"],
                      var_name="item", value_name="score").dropna(subset=["score"])
    melted["is_base"] = melted["iteration"].eq(0)
    for tgt, note, tmap in (("final", "final iteration", None),
                            ("best", "best iteration (own oracle)", BEST)):
        D = stats.item_endpoint_deltas(melted, target_iter_by_arm=tmap,
                                       short=eda_analysis.display_label)
        figd = plots.item_delta_bars(D, title=f"{qname} — which components drive the change ({note})",
                                     xlabel=xlabel)
        if figd:
            eda_analysis.save_fig(figd, f"{slug}_item_deltas_{tgt}",
                                  caption=f"Δ vs base per {qname} component at each arm's {note} "
                                          "(one panel per arm, shared Δ scale; mixed units — see xlabel).")
            plt.show()
        if tgt == "final":
            Tf = D
        else:
            T = pd.concat([Tf.assign(target="final"), D.assign(target="best")], ignore_index=True)
            T = T[["target"] + [c for c in T.columns if c != "target"]]
            eda_analysis.save_table(T, f"{slug}_item_deltas",
                                    caption=f"Per (arm, {qname} component): base/target values + Δ at the "
                                            "final AND best iteration (target column).")

## 1 · Q1 — Session Satisfaction (5 items)  `[EVAL]`
Half of the training reward. The 5 items separate *chat/content satisfaction* from *motivation facilitated* and *learning (relevance)* — the item view shows whether the reward gain is satisfaction-flavored or learning-flavored.

In [ ]:
Q1I = item_section("Q1", "q1")

## 2 · Q2 — Working Alliance / Relational Communication (17 items)  `[EVAL]`
The other half of the training reward, and the reward-composition story: items 1/2/3/10 reward therapist **self-disclosure** — behavior MI does not prescribe. Bars are colored by our face-content groups (analytical, not a validated subscale); the group-trajectory figure shows *when* each exploited component takes off.

In [ ]:
Q2I = item_section("Q2", "q2", q2_style=True)
if not Q2I.empty:
    fig = plots.q2_item_group_trajectory(Q2I)
    if fig:
        eda_analysis.save_fig(fig, "q2_item_group_trajectories",
                              caption="Mean Q2 item-group score across iterations per arm (+-95% CI; face-content groups, analytical not validated) — when each exploited reward component takes off.")
        plt.show()

## 3 · WAI-SR — Working Alliance (3 subscales + 12 items)  `[EVAL]`
Held-out alliance instrument (overlaps Q2 by design — both alliance measures). First the validated **Goal / Task / Bond** subscales, then the full 12-item drill-down.

In [ ]:
SUB = eda_analysis.load_subscales(S.ARMS)
fig = plots.subscale_trajectory_grid(SUB, parents=("WAI-SR",), min_iters=3)
if fig:
    eda_analysis.save_fig(fig, "wai_subscales",
                          caption="WAI-SR Goal/Task/Bond subscale means across iterations; one panel per arm, arms with <3 scored iters omitted.")
    plt.show()
WAII = item_section("WAI-SR", "wai")

## 4 · CSQ-8 — Client Satisfaction (8 items)  `[EVAL]`
Held-out satisfaction instrument — the item view separates service-quality items from the recommend/return intentions.

In [ ]:
CSQI = item_section("CSQ-8", "csq")

## 5 · MI-SAT — MI Intervention Satisfaction (6 items)  `[EVAL]`
Held-out MI-specific satisfaction — `LikelyChange` (item 6) is the closest thing to a patient-outcome item in the halo cluster.

In [ ]:
MISATI = item_section("MI-SAT", "misat")

## 6 · MITI — MI Treatment Integrity (4 globals + 7 behaviour rates + ratios + official thresholds)  `[EVAL]`
The technique instrument. The detail grid is the questionnaire-level home of the old *behaviour drift* figure: the 4 global ratings (1–5), all 7 behaviour counts **per therapist turn** (length-normalized), and the derived proficiency ratios `R:Q`/`%CR`/`%MICO`. **Read:** `B6_AF` (affirmations) ↑ while `B3_Q` (questions) ↓ = the affirmation/advice drift. §6b anchors the summary scores against the **official MITI 4.2.1 competency thresholds** (expert opinion, ~20-min human sessions — out-of-domain caveat applies). Per-metric zooms → `2_questionnaires/miti/`.

In [ ]:
MITI_DET = behavior.miti_detail_by_iter(S.ARMS)
if MITI_DET.empty:
    print("MITI not scored yet — run Run_Eval.ipynb to populate this section.")
else:
    MITI_METRICS = [c for c in MITI_DET.columns if c not in ("arm", "method", "K", "iteration")]
    fig = plots.behavior_trajectory_grid(
        MITI_DET, palette=S.PALETTE, metrics=MITI_METRICS, ncols=4,
        title="MITI detail — global ratings (1–5) + behaviour rates per therapist turn + proficiency ratios")
    if fig:
        eda_analysis.save_fig(fig, "miti_detail_grid", caption="MITI drill-down: 4 global ratings (1-5), all 7 behaviour counts PER THERAPIST TURN (length-normalized), and the derived proficiency ratios R:Q/%CR/%MICO across iterations, all arms. B6_AF up while B3_Q down = the affirmation/advice drift.")
        eda_analysis.save_fig(fig, "miti_detail_grid", group="0_headline", caption="MITI detail grid — headline copy; canonical: 2_questionnaires/.")
        plt.show()
    # Per-metric zoom (rates + RtoQ + Empathy) -> 2_questionnaires/miti/.
    for m in [c for c in MITI_METRICS if c.endswith("_per_turn")] + ["RtoQ", "Empathy"]:
        figm = plots.single_behavior_trajectory(MITI_DET, m, palette=S.PALETTE)
        if figm is not None:
            eda_analysis.save_fig(figm, m, group="2_questionnaires/miti",
                                  caption=f"{eda_analysis.display_label(m)} across iterations, all arms — per-metric zoom of miti_detail_grid.")
            plt.close(figm)
    MT = MITI_DET.round(3).sort_values(["arm", "iteration"]).reset_index(drop=True)
    display(MT)
    eda_analysis.save_table(MT, "miti_detail_by_iter", caption="Mean MITI globals + behaviour rates per therapist turn + proficiency ratios per (arm, iteration), all arms.")
    detail_deltas(MITI_DET, "miti", "MITI",
                  xlabel="Δ vs base (mixed units: globals 1–5 · rates /turn · ratios)")

### 6b · Official MITI 4.2.1 competency thresholds — absolute anchor  `[EVAL]`
Everything else in the EDA is *relative* (vs base, vs the other arm). This anchors the therapist against the **official MITI 4.2.1 standards** (Moyers, Manuel & Ernst 2014, manual rev. 2015, §I): `R:Q` (fair ≥ 1:1, good ≥ 2:1), `%CR` (≥ 40% / 50%), **Technical global** (≥ 3 / 4), **Relational global** (≥ 3.5 / 4). **Read:** training moves both arms from *below basic competence* to **fair-to-good on the global ratings** (Relational crosses "good"), but **neither arm reaches "good" on the technique ratios** — and GRPO's late `R:Q` jump is the *pathological* route (the iter-10 question collapse shrinks the denominator; cross-check the `B3_Q/turn` panel above and `3_Validity_and_Hacking` §2b). **Caveats:** thresholds are expert opinion without normative validation, defined for ~20-min human audio sessions — use as an anchor, not a certification.

In [ ]:
PROF = behavior.miti_proficiency_by_iter(S.ARMS)
fig = plots.miti_threshold_panel(PROF, palette=S.PALETTE)
if fig is not None:
    eda_analysis.save_fig(fig, "miti_proficiency_thresholds", caption="The 4 official MITI 4.2.1 summary scores (R:Q, %CR, Technical global, Relational global) across iterations vs the manual's fair/good competency thresholds (dashed). Both arms cross 'good' on Relational but neither reaches 'good' on the technique ratios; GRPO's late R:Q jump is driven by the iter-10 question collapse (smaller denominator), not more reflection. Thresholds are expert opinion (MITI 4.2.1 manual) and defined for ~20-min human sessions."); plt.show()
    PROF_T = plots.miti_threshold_table(PROF)
    display(PROF_T)
    eda_analysis.save_table(PROF_T, "miti_threshold_verdicts", caption="Base vs final MITI 4.2.1 summary scores per arm with the manual's competency verdict per score (✓good / ✓fair / ✗ below basic competence).")
else:
    print("MITI not scored yet — run Run_Eval.ipynb to populate the proficiency panel.")

## 7 · PCT — Patient Change-Talk (3 patient globals + talk proportions)  `[EVAL]`
The actual MI *target*, scored from the patient side: Importance / Confidence / Readiness (1–5) + the change/sustain/neutral proportions of patient utterances (`% Sustain ↓`). **Read:** patient motivation moves **modestly and mostly for PTO**; GRPO's late regression shows here too (globals dip at iter 9–10). Per-signal zooms → `2_questionnaires/pct/`.

In [ ]:
PCT_BEH = behavior.pct_behavior_by_iter(S.ARMS)
if PCT_BEH.empty:
    print("PCT not scored yet — run Run_Eval.ipynb with QUESTIONNAIRE_FILTER=['PCT'] to populate this.")
else:
    PCT_GLOB = [c for c in ["PCT_Importance", "PCT_Confidence", "PCT_Readiness"] if c in PCT_BEH.columns]
    PCT_PROP = [c for c in PCT_BEH.columns if c.endswith("_prop")]
    PCT_METRICS = PCT_GLOB + ["PCT_ChangeProp"] + PCT_PROP
    fig = plots.behavior_trajectory_grid(
        PCT_BEH, palette=S.PALETTE, metrics=PCT_METRICS,
        title="PCT detail — patient globals (1–5) + patient-utterance proportions")
    if fig:
        eda_analysis.save_fig(fig, "pct_detail_grid", caption="PCT patient-perspective globals (Importance/Confidence/Readiness, 1-5) + change/sustain/neutral proportions of patient utterances across iterations, all arms. The actual MI target scored from the patient side: motivation moves modestly (more for PTO), while GRPO's late-iteration regression appears as a global dip.")
        plt.show()
    for m in PCT_METRICS:
        figm = plots.single_behavior_trajectory(PCT_BEH, m, palette=S.PALETTE)
        if figm is not None:
            eda_analysis.save_fig(figm, m, group="2_questionnaires/pct",
                                  caption=f"{eda_analysis.display_label(m)} across iterations, all arms — per-signal zoom of pct_detail_grid.")
            plt.close(figm)
    PT = PCT_BEH.round(3).sort_values(["arm", "iteration"]).reset_index(drop=True)
    display(PT)
    eda_analysis.save_table(PT, "pct_patient_by_iter", caption="Mean PCT globals + patient-utterance proportions per (arm, iteration), all arms.")
    detail_deltas(PCT_BEH[["arm", "method", "K", "iteration"] + PCT_METRICS], "pct", "PCT",
                  xlabel="Δ vs base (globals 1–5 · proportions 0–1; % Sustain lower = better)")

## 8 · MICI — MI-Inconsistency (severity + 6 behaviour rates; higher = worse)  `[EVAL]`
The negative-valence instrument — *which* harmful move drives the MI-inconsistency rise? Severity global (1–5) + each MI-inconsistent behaviour **per therapist turn**. **Read:** the rise is **almost entirely over-praise (sycophancy)** — `MICI_OverPraise/turn` climbs ~20–35× while confront / warn / judge stay ~0 — confirming the affirmation-drift reading from the *negative* side; it runs away hardest for GRPO at iter 10. A positive Δ in the delta bars = **deterioration**. Per-behaviour zooms → `2_questionnaires/mici/`.

In [ ]:
MICI_BEH = behavior.mici_behavior_by_iter(S.ARMS)
if MICI_BEH.empty:
    print("MICI not scored yet — run Run_Eval.ipynb with QUESTIONNAIRE_FILTER=['MICI'] to populate this.")
else:
    MICI_RATES = [c for c in MICI_BEH.columns if c.endswith("_rate")]  # the 6 per-behaviour rates
    MICI_METRICS = ["MICI_Severity", "MICI_Rate"] + MICI_RATES
    fig = plots.behavior_trajectory_grid(
        MICI_BEH, palette=S.PALETTE, metrics=MICI_METRICS,
        title="MICI detail — severity + per-therapist-turn rates (higher = worse)")
    if fig:
        eda_analysis.save_fig(fig, "mici_detail_grid", caption="MICI severity global (1-5) + total and per-behaviour MI-inconsistent rates PER THERAPIST TURN across iterations, all arms. Higher = worse. Isolates WHICH harmful move drives the MI-inconsistency rise: over-praise dominates (~20-35x), while confront/warn/judge stay ~0 and advise-without-permission sits at a flat background.")
        eda_analysis.save_fig(fig, "mici_detail_grid", group="0_headline", caption="MICI detail grid (sycophancy decomposition) — headline copy; canonical: 2_questionnaires/.")
        plt.show()
    for m in ["MICI_Severity"] + MICI_RATES:
        figm = plots.single_behavior_trajectory(MICI_BEH, m, palette=S.PALETTE)
        if figm is not None:
            eda_analysis.save_fig(figm, m, group="2_questionnaires/mici",
                                  caption=f"{eda_analysis.display_label(m)} across iterations, all arms — per-behaviour zoom of mici_detail_grid.")
            plt.close(figm)
    MT = MICI_BEH.round(3).sort_values(["arm", "iteration"]).reset_index(drop=True)
    display(MT)
    eda_analysis.save_table(MT, "mici_behavior_by_iter", caption="Mean MICI severity + total and per-behaviour MI-inconsistent rates per therapist turn, per (arm, iteration), all arms.")
    detail_deltas(MICI_BEH[["arm", "method", "K", "iteration"] + MICI_METRICS], "mici", "MICI",
                  xlabel="Δ vs base (severity 1–5 · rates /turn) — HIGHER = WORSE, positive Δ = deterioration")

## 9 · Artifact index
Refresh the per-view `results/<VIEW>/INDEX.md` (every notebook ends with this — whichever ran last completes the map).

In [ ]:
print("index ->", eda_analysis.build_index())